In [1]:
import numpy as np   
import pandas as pd

In [2]:
df=pd.read_csv('lalpurja_house_v2_features_ready.csv')

In [3]:
df_new=df.copy()

In [4]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor

# ─────────────────────────────────────────
# DEFINE X AND Y
# Drop raw string columns and target
# ─────────────────────────────────────────
drop_for_model = ['total_price', 'neighborhood', 'municipality']
X = df_new.drop(columns=drop_for_model)
y = np.log1p(df_new['total_price'])

# Drop redundant raw columns
drop_redundant = ['land_size_aana', 'built_up_sqft', 'is_corner_plot']
X = X.drop(columns=drop_redundant)

print(f"Features: {X.shape[1]}")
print(f"Samples:  {X.shape[0]}")

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Scale for linear models
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ─────────────────────────────────────────
# TRAIN ALL MODELS
# ─────────────────────────────────────────
models = {
    'Ridge':             Ridge(alpha=1.0),
    'Lasso':             Lasso(alpha=0.001),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost':           XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM':          lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1),
    'CatBoost':          CatBoostRegressor(n_estimators=100, random_state=42, verbose=0),
    'KNN':               KNeighborsRegressor(n_neighbors=5),
    'SVR':               SVR(kernel='rbf')
}

linear_models = ['Ridge', 'Lasso', 'KNN', 'SVR']
results = []

for name, model in models.items():
    if name in linear_models:
        model.fit(X_train_sc, y_train)
        y_pred = model.predict(X_test_sc)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    results.append({
        'Model':      name,
        'RMSE (log)': round(rmse, 4),
        'MAE (log)':  round(mae, 4),
        'R² Score':   round(r2, 4),
        'Error %':    f"{(np.exp(mae)-1)*100:.1f}%"
    })
    print(f"✅ {name} — R²: {r2:.4f}  Error: {(np.exp(mae)-1)*100:.1f}%")

results_df = pd.DataFrame(results).sort_values('R² Score', ascending=False)
print("\n=== Model Comparison ===")
print(results_df.to_string(index=False))

Features: 42
Samples:  2187
✅ Ridge — R²: 0.6255  Error: 25.7%
✅ Lasso — R²: 0.6284  Error: 25.7%
✅ Random Forest — R²: 0.6113  Error: 25.4%
✅ Gradient Boosting — R²: 0.6286  Error: 24.8%
✅ XGBoost — R²: 0.6064  Error: 25.7%
✅ LightGBM — R²: 0.6056  Error: 24.7%
✅ CatBoost — R²: 0.6399  Error: 24.4%
✅ KNN — R²: 0.5403  Error: 30.6%
✅ SVR — R²: 0.5866  Error: 25.3%

=== Model Comparison ===
            Model  RMSE (log)  MAE (log)  R² Score Error %
         CatBoost      0.3509     0.2180    0.6399   24.4%
Gradient Boosting      0.3563     0.2219    0.6286   24.8%
            Lasso      0.3564     0.2291    0.6284   25.7%
            Ridge      0.3578     0.2290    0.6255   25.7%
    Random Forest      0.3645     0.2263    0.6113   25.4%
          XGBoost      0.3668     0.2290    0.6064   25.7%
         LightGBM      0.3672     0.2207    0.6056   24.7%
              SVR      0.3759     0.2259    0.5866   25.3%
              KNN      0.3964     0.2673    0.5403   30.6%


In [5]:
# ─────────────────────────────────────────
# QUICK FEATURE IMPORTANCE CHECK
# Why: Need to see if new features (bedrooms,
# bathrooms etc) are actually contributing
# If not — they're adding noise not signal
# ─────────────────────────────────────────
cat_check = CatBoostRegressor(n_estimators=200, random_state=42, verbose=0)
cat_check.fit(X_train, y_train)

feature_imp = pd.DataFrame({
    'Feature':    X_train.columns,
    'Importance': cat_check.get_feature_importance()
}).sort_values('Importance', ascending=False)

print("=== Feature Importance ===")
print(feature_imp.to_string(index=False))

# Check new features specifically
new_house_features = ['bedrooms', 'bathrooms', 'kitchens',
                      'living_rooms', 'parking', 'total_floors',
                      'rooms_total', 'bath_per_bed', 'sqft_per_room',
                      'floors_x_land', 'luxury_score', 'parking_premium']
print("\n=== New Housing Features Importance ===")
print(feature_imp[feature_imp['Feature'].isin(new_house_features)].to_string(index=False))

=== Feature Importance ===
                Feature  Importance
          floors_x_land   24.404846
               log_land   10.330514
   neighborhood_encoded   10.073341
       house_size_score    7.049756
neighborhood_x_district    5.042155
        road_width_feet    4.188435
              house_age    3.090340
              bathrooms    2.335304
           total_floors    1.864718
           boudhanath_m    1.690233
    age_condition_score    1.668948
   municipality_encoded    1.603402
       floor_area_ratio    1.592938
           luxury_score    1.464777
           bath_per_bed    1.447401
             furnishing    1.332867
       urban_centrality    1.329350
            ring_road_m    1.299594
          sqft_per_room    1.237112
          bhatbhateni_m    1.160929
               bedrooms    1.146941
          property_type    1.107153
    municipality_x_ward    1.019550
     public_transport_m    0.958826
      comm_road_premium    0.958587
             hospital_m    0.932785
 

In [6]:
# ─────────────────────────────────────────
# TUNE TOP 3 MODELS
# ─────────────────────────────────────────
param_grids = {
    'CatBoost': {
        'model': CatBoostRegressor(random_state=42, verbose=0),
        'params': {
            'iterations':    [500, 1000, 1500, 2000],
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'depth':         [4, 6, 8],
            'l2_leaf_reg':   [1, 3, 5, 9]
        }
    },
    'XGBoost': {
        'model': XGBRegressor(random_state=42, verbosity=0),
        'params': {
            'n_estimators':     [300, 500, 800],
            'learning_rate':    [0.01, 0.05, 0.1],
            'max_depth':        [3, 5, 7],
            'subsample':        [0.7, 0.8, 1.0],
            'colsample_bytree': [0.7, 0.8, 1.0],
            'min_child_weight': [1, 3, 5]
        }
    },
    'Gradient Boosting': {
        'model': GradientBoostingRegressor(random_state=42),
        'params': {
            'n_estimators':  [200, 300, 500],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth':     [3, 5, 7],
            'subsample':     [0.7, 0.8, 1.0],
            'min_samples_leaf': [1, 3, 5]
        }
    }
}

tuned_results = {}

for name, config in param_grids.items():
    print(f"🔍 Tuning {name}...")
    search = RandomizedSearchCV(
        estimator=config['model'],
        param_distributions=config['params'],
        n_iter=50, cv=5, scoring='r2',
        random_state=42, n_jobs=-1
    )
    search.fit(X_train, y_train)
    y_pred = search.best_estimator_.predict(X_test)

    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    tuned_results[name] = {
        'model':  search.best_estimator_,
        'r2':     r2,
        'mae':    mae,
        'rmse':   rmse,
        'params': search.best_params_
    }
    print(f"✅ {name} — R²: {r2:.4f}  Error: {(np.exp(mae)-1)*100:.1f}%")
    print(f"   Best params: {search.best_params_}")

# ─────────────────────────────────────────
# PICK BEST AND SAVE
# ─────────────────────────────────────────
best_name = max(tuned_results, key=lambda x: tuned_results[x]['r2'])
best      = tuned_results[best_name]

print(f"\n=== Best Model: {best_name} ===")
print(f"R²:     {best['r2']:.4f}")
print(f"Error:  {(np.exp(best['mae'])-1)*100:.1f}%")
print(f"RMSE:   {best['rmse']:.4f}")

# Save model and scaler
import joblib
joblib.dump(best['model'], 'catboost_lalpurja_house_v2_final.pkl')
joblib.dump(scaler,        'scaler_lalpurja_house_v2.pkl')

# Verify predictions
loaded = joblib.load('catboost_lalpurja_house_v2_final.pkl')
sample_preds = np.expm1(loaded.predict(X_test[:5]))
sample_actual = np.expm1(y_test[:5])

print(f"\n✅ Saved catboost_lalpurja_house_v2_final.pkl")
print(f"\n=== Sample Predictions vs Actual ===")
for pred, actual in zip(sample_preds, sample_actual):
    diff = abs(pred - actual) / actual * 100
    print(f"  Predicted: {pred/1e6:.2f}Cr  |  "
          f"Actual: {actual/1e6:.2f}Cr  |  "
          f"Diff: {diff:.1f}%")

print(f"\n=== FINAL MODEL SUMMARY ===")
print(f"Algorithm:     {best_name}")
print(f"R² Score:      {best['r2']:.4f}")
print(f"Average Error: {(np.exp(best['mae'])-1)*100:.1f}%")
print(f"Training rows: {X_train.shape[0]}")
print(f"Features:      {X_train.shape[1]}")

🔍 Tuning CatBoost...
✅ CatBoost — R²: 0.6455  Error: 23.7%
   Best params: {'learning_rate': 0.05, 'l2_leaf_reg': 5, 'iterations': 1000, 'depth': 6}
🔍 Tuning XGBoost...
✅ XGBoost — R²: 0.6355  Error: 24.2%
   Best params: {'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
🔍 Tuning Gradient Boosting...
✅ Gradient Boosting — R²: 0.6172  Error: 24.5%
   Best params: {'subsample': 0.8, 'n_estimators': 200, 'min_samples_leaf': 5, 'max_depth': 5, 'learning_rate': 0.05}

=== Best Model: CatBoost ===
R²:     0.6455
Error:  23.7%
RMSE:   0.3481

✅ Saved catboost_lalpurja_house_v2_final.pkl

=== Sample Predictions vs Actual ===
  Predicted: 35.05Cr  |  Actual: 33.00Cr  |  Diff: 6.2%
  Predicted: 121.66Cr  |  Actual: 110.00Cr  |  Diff: 10.6%
  Predicted: 15.38Cr  |  Actual: 15.00Cr  |  Diff: 2.6%
  Predicted: 35.85Cr  |  Actual: 38.00Cr  |  Diff: 5.7%
  Predicted: 46.21Cr  |  Actual: 53.50Cr  |  Diff: 13.6%

=== FINAL MOD